# 2 Gen Reviews

This notebook runs the redesigned review-generation stage for the study.

It generates only original AI NCEMS-criteria reviews for the human proposals.
It does not run novelty reviews, rephrasing, score aggregation, or downstream sampling.

In [1]:
REVIEW_CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
MODELS_TO_USE = ['gpt-5.5', 'gemini-3.1-pro-preview', 'claude-sonnet-5']

REVIEWS_PER_MODEL_PER_PROPOSAL = 5
GENERATION_TEMPERATURE = 0.9
MAX_TOKENS_REVIEWS = 12000
RETRY_DELAYS = [2, 5, 10]
SAVE_PROGRESS_EVERY_N_CALLS = 10
RESUME_OK = True

RUN_TEST_CALLS = False
TEST_MODEL = MODELS_TO_USE[0]
TEST_MAX_TOKENS = MAX_TOKENS_REVIEWS

# Set to a fixed string only if you want a custom run id.
RUN_ID = None


In [2]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager
from review_generation import (
    build_persona_review_schedule,
    build_review_condition_registry,
    build_review_schedule,
    find_project_root,
    load_human_proposal_roster,
    load_human_review_counts,
    load_reviewer_persona_roster,
    load_shared_call_context,
    make_review_prompt,
    parse_review_response,
    run_review_generation_for_condition,
)
from proposal_generation import now_run_id

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
shared_review_context = load_shared_call_context(PROJECT_ROOT)
human_review_target_roster = load_human_proposal_roster(PROJECT_ROOT)
human_review_counts = load_human_review_counts(PROJECT_ROOT)
human_review_target_roster = human_review_target_roster.merge(
    human_review_counts,
    on=['target_cohort', 'target_proposal_id', 'target_proposal_uid'],
    how='left',
    validate='one_to_one',
)
if human_review_target_roster['target_human_n_reviews'].isna().any():
    missing = human_review_target_roster.loc[
        human_review_target_roster['target_human_n_reviews'].isna(),
        ['target_proposal_uid', 'target_proposal_title']
    ]
    raise RuntimeError(f'Missing human review counts for proposals: {missing.to_dict(orient="records")}')
 
reviewer_persona_roster = load_reviewer_persona_roster(PROJECT_ROOT, human_review_target_roster)
reviewer_persona_schedule = build_persona_review_schedule(
    human_review_target_roster,
    reviewer_persona_roster,
)
review_condition_registry = build_review_condition_registry()
prompt_manager = PromptManager()
ai_interface = AIModelsInterface(config_path=str(PROJECT_ROOT / '.env'), override_env=True)
available_models = ai_interface.get_available_models()
resolved_models = [ai_interface.resolve_model_name(model_name) for model_name in MODELS_TO_USE]
missing_models = [model_name for model_name in resolved_models if model_name not in available_models]
if missing_models:
    raise RuntimeError(f'Requested model(s) unavailable with current API keys: {missing_models}')
if REVIEWS_PER_MODEL_PER_PROPOSAL != 5:
    raise RuntimeError('This redesigned notebook currently assumes 5 reviews per model per proposal.')

stage_run_id = RUN_ID or now_run_id()

print(f'Project root: {PROJECT_ROOT}')
print(f'Human target proposals: {len(human_review_target_roster)}')
print('Human review count distribution:')
print(human_review_target_roster['target_human_n_reviews'].value_counts().sort_index())
print(f'Available canonical models: {available_models}')
print(f'Review conditions to run: {REVIEW_CONDITIONS_TO_RUN}')

INFO:ai_models_interface:OpenAI GPT-5.5 initialized
INFO:ai_models_interface:Google Gemini initialized
INFO:ai_models_interface:Anthropic Claude initialized


Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal
Human target proposals: 23
Human review count distribution:
target_human_n_reviews
2     1
3     6
4    15
5     1
Name: count, dtype: int64
Available canonical models: ['claude-sonnet-5', 'gemini-3.1-pro-preview', 'gpt-5.5']
Review conditions to run: ['baseline', 'one_at_a_time', 'persona']


## Optional Test Calls

Set `RUN_TEST_CALLS = True` to make one sample review-generation call per condition before the full schedule runs. This lets you inspect the exact prompt, raw response, and parsed structured reviews for one target proposal/model pair.


In [3]:
review_test_call_outputs = {}

if not RUN_TEST_CALLS:
    print('RUN_TEST_CALLS is False; skipping sample review-generation calls.')
else:
    test_model = ai_interface.resolve_model_name(TEST_MODEL)
    if test_model not in resolved_models:
        raise RuntimeError(f'TEST_MODEL must be one of MODELS_TO_USE after alias resolution: {resolved_models}')

    for condition in REVIEW_CONDITIONS_TO_RUN:
        schedule_df = build_review_schedule(
            target_roster=human_review_target_roster,
            condition_config=review_condition_registry[condition],
            models_to_use=[test_model],
            reviewer_persona_schedule=reviewer_persona_schedule if condition == 'persona' else None,
        )
        sample_row = schedule_df.iloc[0].to_dict()
        prompt, target_text_truncated = make_review_prompt(
            prompt_manager=prompt_manager,
            schedule_row=sample_row,
            shared_review_context=shared_review_context,
        )
        result = ai_interface.generate_content_with_metadata(
            prompt,
            model_name=test_model,
            temperature=GENERATION_TEMPERATURE,
            max_tokens=TEST_MAX_TOKENS,
            retry_delays=RETRY_DELAYS,
        )
        parsed_reviews, parse_error = parse_review_response(
            result['raw_response'],
            expected_count=int(sample_row['expected_review_count']),
        ) if not result['error'] else ([], None)

        review_test_call_outputs[condition] = {
            'condition': condition,
            'model': test_model,
            'prompt_template': sample_row['review_prompt_template'],
            'sample_call_id': sample_row['review_call_id'],
            'expected_review_count': int(sample_row['expected_review_count']),
            'target_proposal_uid': sample_row['target_proposal_uid'],
            'target_proposal_title': sample_row['target_proposal_title'],
            'target_text_truncated': target_text_truncated,
            'prompt': prompt,
            'raw_response': result['raw_response'],
            'provider_model_id': result.get('provider_model_id', ''),
            'timestamp': result.get('timestamp', ''),
            'error': result.get('error', ''),
            'parse_error': parse_error,
            'parsed_reviews': parsed_reviews,
        }

    review_test_summary = pd.DataFrame([
        {
            'condition': payload['condition'],
            'model': payload['model'],
            'target_proposal_uid': payload['target_proposal_uid'],
            'prompt_template': payload['prompt_template'],
            'expected_review_count': payload['expected_review_count'],
            'parsed_review_count': len(payload['parsed_reviews']),
            'target_text_truncated': payload['target_text_truncated'],
            'error': payload['error'],
            'parse_error': payload['parse_error'],
        }
        for payload in review_test_call_outputs.values()
    ])
    display(review_test_summary)

    for condition, payload in review_test_call_outputs.items():
        print(f'\n=== Test Call: {condition} ===')
        print(f"Target proposal: {payload['target_proposal_uid']} | {payload['target_proposal_title']}")
        print('Prompt:')
        print(payload['prompt'])
        print('\nRaw response:')
        print(payload['raw_response'])
        print('\nParsed reviews:')
        display(pd.DataFrame(payload['parsed_reviews']))


RUN_TEST_CALLS is False; skipping sample review-generation calls.


In [4]:
review_generation_outputs = {}
schedule_registry = {}

for condition in REVIEW_CONDITIONS_TO_RUN:
    print(f'\n=== Review Generation: {condition} ===')
    schedule_df = build_review_schedule(
        target_roster=human_review_target_roster,
        condition_config=review_condition_registry[condition],
        models_to_use=resolved_models,
        reviewer_persona_schedule=reviewer_persona_schedule if condition == 'persona' else None,
    )
    schedule_registry[condition] = schedule_df

    condition_result = run_review_generation_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        prompt_manager=prompt_manager,
        shared_review_context=shared_review_context,
        condition_config=review_condition_registry[condition],
        schedule_df=schedule_df,
        generation_temperature=GENERATION_TEMPERATURE,
        max_tokens=MAX_TOKENS_REVIEWS,
        retry_delays=RETRY_DELAYS,
        save_progress_every_n_calls=SAVE_PROGRESS_EVERY_N_CALLS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    review_generation_outputs[condition] = condition_result
    print(f"Schedule file: {condition_result['schedule_path']}")
    print(f"Complete file: {condition_result['complete_path']}")
    print(f"Rows: {len(condition_result['reviews_df'])}")
    if condition_result['qa_issues']:
        print('QA issues:')
        for issue in condition_result['qa_issues'][:10]:
            print(f'  - {issue}')



=== Review Generation: baseline ===


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:httpx:HTTP Reques

Schedule file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/baseline/review_schedule_baseline_20260709_143810.csv
Complete file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/baseline/ai_reviews_baseline_complete_20260709_143810.csv
Rows: 330
QA issues:
  - baseline: expected 345 completed review rows, found 330

=== Review Generation: one_at_a_time ===


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 502 Bad Gateway"
INFO:openai._base_client:Retrying request to /chat/completions in 60.000000 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:

Schedule file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/one_at_a_time/review_schedule_one_at_a_time_20260709_143810.csv
Complete file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/one_at_a_time/ai_reviews_one_at_a_time_complete_20260709_143810.csv
Rows: 331
QA issues:
  - one_at_a_time: expected 345 completed review rows, found 331
  - one_at_a_time: y1::1 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - one_at_a_time: y1::11 x claude-sonnet-5 has 4 review rows; expected 5
  - one_at_a_time: y1::11 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - one_at_a_time: y1::12 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - one_at_a_time: y1::3 x claude-sonnet-5 has 4 review rows; expected 5
  - one_at_a_time: y1::3 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - one_at_a_time: y1::4 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - one_at_a_time: y1::7 x gemini-3.1-pro-preview has 2

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:root:AFC is enabled with max remote calls: 10.
INFO:root:AFC remote call 1 is done.
INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:httpx:HTT

Schedule file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/persona/review_schedule_persona_20260709_143810.csv
Complete file: /Users/eveyhuang/Documents/NICO/human-AI-proposal/data/reviews/ai_reviews/persona/ai_reviews_persona_complete_20260709_143810.csv
Rows: 329
QA issues:
  - persona: expected 345 completed review rows, found 329
  - persona: y1::10 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - persona: y1::11 x claude-sonnet-5 has 4 review rows; expected 5
  - persona: y1::12 x claude-sonnet-5 has 4 review rows; expected 5
  - persona: y1::12 x gemini-3.1-pro-preview has 3 review rows; expected 5
  - persona: y1::2 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - persona: y1::4 x gemini-3.1-pro-preview has 4 review rows; expected 5
  - persona: y1::5 x gemini-3.1-pro-preview has 3 review rows; expected 5
  - persona: y1::7 x claude-sonnet-5 has 4 review rows; expected 5
  - persona: y1::7 x gemini-3.1-pro-preview has 4 review ro

In [5]:
summary_rows = []
for condition in REVIEW_CONDITIONS_TO_RUN:
    result = review_generation_outputs[condition]
    reviews_df = result['reviews_df']
    summary_rows.append(
        {
            'condition': condition,
            'schedule_rows': len(schedule_registry[condition]),
            'review_rows': len(reviews_df),
            'target_proposals': reviews_df['target_proposal_uid'].nunique() if not reviews_df.empty else 0,
            'models': ', '.join(sorted(reviews_df['evaluator_model'].dropna().unique())) if not reviews_df.empty else '',
            'distinct_call_ids': reviews_df['review_call_id'].nunique() if not reviews_df.empty else 0,
            'complete_file': str(result['complete_path']) if result['complete_path'] else '',
            'reused_existing': result['reused_existing'],
            'qa_issue_count': len(result['qa_issues']),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df


,condition,schedule_rows,review_rows,target_proposals,models,distinct_call_ids,complete_file,reused_existing,qa_issue_count
0,baseline,69,330,23,"claude-sonnet-5, gemini-3.1-pro-preview, gpt-5.5",66,/Users/eveyhuang/Documents/NICO/human-AI-propo...,False,1
1,one_at_a_time,345,331,23,"claude-sonnet-5, gemini-3.1-pro-preview, gpt-5.5",331,/Users/eveyhuang/Documents/NICO/human-AI-propo...,False,23
2,persona,345,329,23,"claude-sonnet-5, gemini-3.1-pro-preview, gpt-5.5",329,/Users/eveyhuang/Documents/NICO/human-AI-propo...,False,29
